# 🚀 MASK R-CNN TRAINING ON KAGGLE GPU

**Organized Training Pipeline with Auto-Save Protection**

This notebook is optimized for:
- ✅ Kaggle GPU training  
- ✅ **Automatic checkpointing every 400 iterations**
- ✅ **Continuous backup to Kaggle output**
- ✅ Resume training from any checkpoint
- ✅ No data loss even if session crashes

**Run cells in order from top to bottom** ⬇️

---
## ⚠️ BEFORE STARTING: Prepare Your Dataset on Kaggle

**You need to upload your dataset to Kaggle first!**

### Option A: Create a Kaggle Dataset

1. Go to https://www.kaggle.com/datasets
2. Click **"New Dataset"**
3. Upload your **INSEG_PRO** folder containing:
   ```
   INSEG_PRO/
   ├── data_zips/
   │   ├── leftImg8bit_trainvaltest.zip
   │   └── gtFine_trainvaltest.zip
   └── code/
       └── dataset_loader.py
   ```
4. Name it **"INSEG_PRO"** (or any name you prefer)
5. Click **"Create"**

### Option B: Add Existing Dataset

1. In this notebook, click **"Add data"** (right sidebar)
2. Search for your uploaded dataset
3. Add it to the notebook

### Expected Structures:

The notebook handles both flat and nested structures:

**Flat structure:**
```
/kaggle/input/YOUR-DATASET/
├── data_zips/
│   ├── leftImg8bit_trainvaltest.zip
│   └── gtFine_trainvaltest.zip
└── code/
    └── dataset_loader.py
```

**Nested structure (your case):**
```
/kaggle/input/INSEG_PRO/
└── INSEG_PRO/
    ├── data_zips/
    │   ├── leftImg8bit_trainvaltest.zip
    │   └── gtFine_trainvaltest.zip
    └── code/
        └── dataset_loader.py
```

**Note:** Step 2 will auto-detect your structure! ✅

---
## 📋 STEP 1: Environment Setup & GPU Check

### ⚠️ ALTERNATIVE: If Step 1 Installation Fails

**If the installation below fails, skip it and run this alternative cell instead.**

This method clones Detectron2 locally and builds it from source.

In [ ]:
# ========================================
# ALTERNATIVE DETECTRON2 INSTALLATION
# ========================================
# Only run this if the main installation (Step 1) failed!

import torch
import sys
import os
from pathlib import Path

print("🔄 Alternative Detectron2 Installation Method")
print("=" * 70)

# Install dependencies
print("📦 Installing dependencies...")
!pip install -q --upgrade pip setuptools wheel
!pip install -q cython opencv-python pillow matplotlib pycocotools

# Method 1: Try pre-built wheel (fastest)
print("\n🔄 Trying pre-built wheel installation...")
try:
    # Get PyTorch and CUDA versions
    torch_version = torch.__version__.split('+')[0]
    torch_major_minor = '.'.join(torch_version.split('.')[:2])
    
    if torch.cuda.is_available():
        cuda_version = torch.version.cuda
        cuda_short = cuda_version.replace('.', '')[:4]  # e.g., "118" from "11.8"
        
        # Try pre-built wheel URL
        wheel_url = f"https://dl.fbaipublicfiles.com/detectron2/wheels/cu{cuda_short}/torch{torch_major_minor}/index.html"
        print(f"   Trying: {wheel_url}")
        !pip install -q detectron2 -f {wheel_url}
    else:
        # CPU version
        !pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cpu/torch{torch_major_minor}/index.html
    
    # Test if it worked
    import detectron2
    print(f"✅ Pre-built wheel installed successfully!")
    print(f"   Version: {detectron2.__version__}")
    
except Exception as e:
    print(f"   Pre-built wheel failed: {e}")
    print("\n🔄 Trying source installation...")
    
    # Method 2: Install from source (fallback)
    try:
        # Clone repository
        print("📦 Downloading Detectron2 source...")
        detectron2_path = Path("/tmp/detectron2_install")
        
        if detectron2_path.exists():
            !rm -rf {str(detectron2_path)}
        
        !git clone --depth 1 https://github.com/facebookresearch/detectron2.git {str(detectron2_path)}
        
        # Verify clone succeeded
        if not detectron2_path.exists():
            raise FileNotFoundError("Git clone failed!")
        
        print(f"✅ Source downloaded to: {detectron2_path}")
        
        # Build and install
        print("🔨 Building from source (takes 3-5 minutes)...")
        original_dir = os.getcwd()
        os.chdir(str(detectron2_path))
        
        !python -m pip install -e . --no-build-isolation
        
        os.chdir(original_dir)
        sys.path.insert(0, str(detectron2_path))
        
        print("✅ Built from source successfully!")
        
    except Exception as e2:
        print(f"\n❌ Source installation also failed: {e2}")
        print("\n🔄 Final attempt: Direct pip install...")
        
        # Method 3: Last resort
        !pip install detectron2
        
# Final verification
print("\n" + "=" * 70)
print("🔍 FINAL VERIFICATION")
print("=" * 70)

try:
    import detectron2
    from detectron2 import model_zoo
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultTrainer
    
    print(f"✅ Detectron2 successfully installed!")
    print(f"   Version: {detectron2.__version__}")
    print(f"   Location: {detectron2.__file__}")
    print("=" * 70)
    
except ImportError as e:
    print(f"❌ Installation failed: {e}")
    print("\n📝 Manual installation required:")
    print("   1. Check your Kaggle/Colab environment")
    print("   2. Ensure GPU is enabled")
    print("   3. Try restarting the kernel")
    print("=" * 70)
    raise

In [ ]:
# ========================================
# STEP 1: CHECK GPU & INSTALL DETECTRON2
# ========================================

import torch
import sys
import os

print("=" * 70)
print("🔍 CHECKING ENVIRONMENT")
print("=" * 70)

# Check GPU
if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ NO GPU DETECTED!")
    print("   Kaggle: Settings > Accelerator > GPU T4 x2")
    print("   Enable GPU in notebook settings!")
    # Don't exit, just warn
    print("\n⚠️  WARNING: Training will be very slow without GPU!")

print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

# Install Detectron2 - Simple & Robust Method for Kaggle
print("\n📦 Installing Detectron2...")
print("   This may take 3-5 minutes...")

# Install dependencies first
print("   Installing dependencies...")
!pip install -q --upgrade pip
!pip install -q opencv-python pillow pycocotools matplotlib

# Install detectron2 from source without build isolation
print("   Installing detectron2...")
!python -m pip install --no-build-isolation -q 'git+https://github.com/facebookresearch/detectron2.git'

# Verify installation
print("\n🔍 Verifying installation...")
try:
    import detectron2
    from detectron2 import model_zoo
    from detectron2.config import get_cfg
    print(f"✅ Detectron2 installed successfully!")
    print(f"   Version: {detectron2.__version__}")
except ImportError as e:
    print(f"\n❌ Detectron2 import failed: {e}")
    print("\n📝 Please use the ALTERNATIVE installation method below:")
    print("   (See the markdown cell above for alternative code)")
    raise

print("=" * 70)

---
## 📂 STEP 2: Setup Kaggle Paths

In [ ]:
# ========================================
# STEP 2: SETUP KAGGLE PATHS (AUTO-DETECT)
# ========================================

from pathlib import Path
import os

# Auto-detect Kaggle input directory
kaggle_inputs = list(Path("/kaggle/input").glob("*"))
print("=" * 70)
print("🔍 DETECTING KAGGLE DATASET")
print("=" * 70)
print(f"Available datasets in /kaggle/input:")
for inp in kaggle_inputs:
    print(f"   - {inp.name}")

# Try to find dataset automatically
# Option 1: Try common names
possible_names = ["INSEG_PRO", "inseg-dataset", "cityscapes-inseg"]
KAGGLE_INPUT = None

for name in possible_names:
    test_path = Path(f"/kaggle/input/{name}")
    if test_path.exists():
        KAGGLE_INPUT = test_path
        print(f"\n✅ Found dataset: {name}")
        break

# Option 2: If not found, try first dataset
if KAGGLE_INPUT is None and kaggle_inputs:
    KAGGLE_INPUT = kaggle_inputs[0]
    print(f"\n⚠️  Using first available dataset: {KAGGLE_INPUT.name}")

# Option 3: Manual override (uncomment and edit if needed)
# KAGGLE_INPUT = Path("/kaggle/input/YOUR-DATASET-NAME-HERE")

# Verify dataset exists
if KAGGLE_INPUT is None or not KAGGLE_INPUT.exists():
    print("\n❌ ERROR: No dataset found!")
    print("\nPlease:")
    print("1. Upload your dataset to Kaggle")
    print("2. Add it to this notebook (Click 'Add data' button)")
    print("3. Uncomment and edit KAGGLE_INPUT path above")
    raise FileNotFoundError("Dataset not found in /kaggle/input/")

# Handle nested structure (INSEG_PRO/INSEG_PRO/)
print(f"\n🔍 Exploring dataset structure...")
def find_data_code_dirs(root_path):
    """Recursively search for data_zips and code folders"""
    data_zips_dir = None
    code_dir = None
    
    # Check current level
    if (root_path / "data_zips").exists():
        data_zips_dir = root_path / "data_zips"
    if (root_path / "code").exists():
        code_dir = root_path / "code"
    
    # If found both, return
    if data_zips_dir and code_dir:
        return data_zips_dir, code_dir
    
    # Otherwise, search one level deeper (handle nested INSEG_PRO/INSEG_PRO/)
    for subdir in root_path.iterdir():
        if subdir.is_dir():
            if not data_zips_dir and (subdir / "data_zips").exists():
                data_zips_dir = subdir / "data_zips"
            if not code_dir and (subdir / "code").exists():
                code_dir = subdir / "code"
            
            if data_zips_dir and code_dir:
                return data_zips_dir, code_dir
    
    return data_zips_dir, code_dir

DATA_ZIPS_DIR, CODE_DIR = find_data_code_dirs(KAGGLE_INPUT)

# Verify we found the folders
if DATA_ZIPS_DIR is None or CODE_DIR is None:
    print("\n❌ ERROR: Could not find data_zips and code folders!")
    print(f"\nSearched in: {KAGGLE_INPUT}")
    print("\nDataset structure should be:")
    print("   /kaggle/input/YOUR-DATASET/")
    print("   ├── data_zips/")
    print("   └── code/")
    print("\nOR (nested):")
    print("   /kaggle/input/YOUR-DATASET/INSEG_PRO/")
    print("   ├── data_zips/")
    print("   └── code/")
    raise FileNotFoundError("Required folders not found!")

# Define working directories
DATA_DIR = Path("/kaggle/working/data/raw")
OUTPUT_DIR = Path("/kaggle/working/output")
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")

# Add code to path
sys.path.insert(0, str(CODE_DIR))

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("✅ PATHS CONFIGURED")
print("=" * 70)
print(f"📁 Kaggle input: {KAGGLE_INPUT}")
print(f"📁 Data ZIPs: {DATA_ZIPS_DIR}")
print(f"📁 Code dir: {CODE_DIR}")
print(f"📁 Dataset (working): {DATA_DIR}")
print(f"📁 Training output: {OUTPUT_DIR}")
print(f"📁 Checkpoints: {CHECKPOINT_DIR}")
print("=" * 70)

# Verify dataset structure
print("\n🔍 Verifying dataset contents...")
if DATA_ZIPS_DIR.exists():
    zip_files = list(DATA_ZIPS_DIR.glob("*.zip"))
    if zip_files:
        print(f"✅ Found {len(zip_files)} ZIP file(s):")
        for zf in zip_files:
            size_mb = zf.stat().st_size / (1024 * 1024)
            print(f"   - {zf.name} ({size_mb:.1f} MB)")
    else:
        print(f"⚠️  No ZIP files in: {DATA_ZIPS_DIR}")
else:
    print(f"❌ data_zips folder not found!")
    
if CODE_DIR.exists():
    py_files = list(CODE_DIR.glob("*.py"))
    if py_files:
        print(f"✅ Found {len(py_files)} Python file(s):")
        for pf in py_files:
            print(f"   - {pf.name}")
    else:
        print(f"⚠️  No Python files in: {CODE_DIR}")
else:
    print(f"❌ code folder not found!")

print("=" * 70)

---
## 📦 STEP 3: Extract Dataset from ZIPs

In [ ]:
# ========================================
# STEP 3: EXTRACT DATASET (5-10 minutes)
# ========================================

import zipfile
import time

print("=" * 70)
print("📦 EXTRACTING CITYSCAPES DATASET")
print("=" * 70)
print("⏱️  This takes 5-10 minutes (only once per session)\n")

DATA_DIR.mkdir(parents=True, exist_ok=True)
start_time = time.time()

# Extract images
leftimg_zip = DATA_ZIPS_DIR / "leftImg8bit_trainvaltest.zip"
if leftimg_zip.exists():
    print("🔄 Extracting images (leftImg8bit)...")
    with zipfile.ZipFile(leftimg_zip, 'r') as zip_ref:
        # Show progress
        members = zip_ref.namelist()
        for i, member in enumerate(members):
            zip_ref.extract(member, DATA_DIR)
            if i % 1000 == 0:
                print(f"   Progress: {i}/{len(members)} files...")
    print("   ✅ Images extracted")
else:
    print(f"❌ Not found: {leftimg_zip}")
    raise FileNotFoundError("Upload leftImg8bit_trainvaltest.zip to Kaggle dataset!")

# Extract annotations
gtfine_zip = DATA_ZIPS_DIR / "gtFine_trainvaltest.zip"
if gtfine_zip.exists():
    print("\n🔄 Extracting annotations (gtFine)...")
    with zipfile.ZipFile(gtfine_zip, 'r') as zip_ref:
        members = zip_ref.namelist()
        for i, member in enumerate(members):
            zip_ref.extract(member, DATA_DIR)
            if i % 500 == 0:
                print(f"   Progress: {i}/{len(members)} files...")
    print("   ✅ Annotations extracted")
else:
    print(f"❌ Not found: {gtfine_zip}")
    raise FileNotFoundError("Upload gtFine_trainvaltest.zip to Kaggle dataset!")

elapsed = time.time() - start_time
print(f"\n✅ Extraction completed in {elapsed/60:.1f} minutes")
print(f"📁 Dataset ready at: {DATA_DIR}")
print("=" * 70)

---
## 📚 STEP 4: Import Libraries

In [ ]:
# ========================================
# STEP 4: IMPORT ALL REQUIRED LIBRARIES
# ========================================

import numpy as np
import json
import random
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import shutil

# Detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor, DefaultTrainer, HookBase
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.structures import BoxMode
from detectron2.utils.events import get_event_storage
from detectron2.evaluation import COCOEvaluator
import detectron2.utils.comm as comm

print("✅ All libraries imported successfully!")

---
## 🗂️ STEP 5: Load Cityscapes Dataset

In [ ]:
# ========================================
# STEP 5: LOAD CITYSCAPES DATASET
# ========================================

from dataset_loader import register_cityscapes_dataset, CITYSCAPES_THING_CLASSES

print("=" * 70)
print("📂 LOADING CITYSCAPES DATASET")
print("=" * 70)

# Register train and validation datasets
train_dataset_name, train_dicts, train_metadata = register_cityscapes_dataset(
    DATA_DIR, split="train"
)
val_dataset_name, val_dicts, val_metadata = register_cityscapes_dataset(
    DATA_DIR, split="val"
)

print(f"\n✅ Dataset loaded successfully!")
print(f"   Training images: {len(train_dicts)}")
print(f"   Validation images: {len(val_dicts)}")
print(f"   Classes: {CITYSCAPES_THING_CLASSES}")
print(f"   Number of classes: {len(CITYSCAPES_THING_CLASSES)}")
print("=" * 70)

---
## ⚙️ STEP 6: Configure Mask R-CNN Model

**Key Features:**
- ✅ Automatic checkpoint every 400 iterations
- ✅ Backup to working directory (Kaggle output)
- ✅ 10,000 iterations for best results
- ✅ Validation every 500 iterations
- ✅ Never lose your progress!

In [ ]:
# ========================================
# STEP 6: CONFIGURE MASK R-CNN WITH AUTO-SAVE
# ========================================

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
)

# ==================== DATASET ====================
cfg.DATASETS.TRAIN = (train_dataset_name,)
cfg.DATASETS.TEST = (val_dataset_name,)
cfg.DATALOADER.NUM_WORKERS = 4

# ==================== MODEL ====================
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)
cfg.MODEL.DEVICE = "cuda"
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CITYSCAPES_THING_CLASSES)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

# ==================== SOLVER ====================
cfg.SOLVER.IMS_PER_BATCH = 4  # Batch size
cfg.SOLVER.BASE_LR = 0.0001  # Learning rate
cfg.SOLVER.MAX_ITER = 10000  # Total iterations
cfg.SOLVER.STEPS = (6000, 8500)  # LR decay steps
cfg.SOLVER.GAMMA = 0.1  # LR decay factor
cfg.SOLVER.WARMUP_ITERS = 500  # Warmup
cfg.SOLVER.WARMUP_FACTOR = 0.001

# ==================== CHECKPOINTING (CRITICAL!) ====================
cfg.SOLVER.CHECKPOINT_PERIOD = 400  # Save every 400 iterations

# ==================== EVALUATION ====================
cfg.TEST.EVAL_PERIOD = 500  # Validate every 500 iterations

# ==================== DATA AUGMENTATION ====================
cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)
cfg.INPUT.MAX_SIZE_TRAIN = 1333
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.INPUT.RANDOM_FLIP = "horizontal"

# ==================== OUTPUT ====================
cfg.OUTPUT_DIR = str(OUTPUT_DIR)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("✅ MODEL CONFIGURATION CREATED")
print("=" * 70)
print(f"🎯 Device: {cfg.MODEL.DEVICE}")
print(f"🎯 Batch size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"🎯 Max iterations: {cfg.SOLVER.MAX_ITER:,}")
print(f"🎯 Learning rate: {cfg.SOLVER.BASE_LR}")
print(f"🎯 Checkpoint every: {cfg.SOLVER.CHECKPOINT_PERIOD} iterations ⚡")
print(f"🎯 Validation every: {cfg.TEST.EVAL_PERIOD} iterations")
print(f"🎯 Output: {cfg.OUTPUT_DIR}")
print("=" * 70)
print("\n⏱️  Estimated training time: 2-3 hours on T4 GPU")
print("=" * 70)

---
## 🔧 STEP 7: Define Custom Trainer with Auto-Backup

**This trainer automatically:**
- 💾 Saves checkpoint every 400 iterations to output directory
- ☁️ Backs up to checkpoints folder every 400 iterations
- 📊 Tracks all losses
- ✅ Validates on validation set
- 🛡️ Protects against session crashes

In [ ]:
# ========================================
# STEP 7: CUSTOM TRAINER WITH AUTO-BACKUP
# ========================================

class CheckpointBackupHook(HookBase):
    """
    Automatically backup checkpoints to checkpoints directory
    Runs every 400 iterations to save your progress
    """
    def __init__(self, checkpoint_dir, backup_period=400):
        super().__init__()
        self.checkpoint_dir = Path(checkpoint_dir)
        self.backup_period = backup_period
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    def after_step(self):
        """Backup checkpoint after each checkpoint period"""
        if (self.trainer.iter + 1) % self.backup_period == 0:
            iteration = self.trainer.iter + 1
            checkpoint_file = Path(self.trainer.cfg.OUTPUT_DIR) / f"model_{iteration:07d}.pth"
            
            if checkpoint_file.exists():
                backup_path = self.checkpoint_dir / f"model_{iteration:07d}.pth"
                print(f"\n💾 Backing up checkpoint: {backup_path.name}")
                shutil.copy(checkpoint_file, backup_path)
                print(f"✅ Backup complete!")


class LossTrackerHook(HookBase):
    """Track and display losses during training"""
    def __init__(self):
        super().__init__()
        self.losses = []
    
    def after_step(self):
        storage = get_event_storage()
        loss_dict = {}
        
        for k, v in storage.latest().items():
            if "loss" in k:
                loss_dict[k] = v[0] if isinstance(v, tuple) else v
        
        if loss_dict:
            self.losses.append(loss_dict)
            
            # Print every 50 iterations
            if self.trainer.iter % 50 == 0:
                total_loss = sum(loss_dict.values())
                print(f"[Iter {self.trainer.iter:5d}] Loss: {total_loss:.4f}")


class ImprovedTrainer(DefaultTrainer):
    """
    Enhanced Trainer with:
    - Automatic checkpoint backup
    - Loss tracking
    - Validation evaluation
    """
    def __init__(self, cfg):
        self.loss_hook = LossTrackerHook()
        self.backup_hook = CheckpointBackupHook(CHECKPOINT_DIR, backup_period=400)
        super().__init__(cfg)
    
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, False, output_folder)
    
    def build_hooks(self):
        hooks = super().build_hooks()
        # Insert custom hooks before periodic writer
        hooks.insert(-1, self.loss_hook)
        hooks.insert(-1, self.backup_hook)
        return hooks

print("=" * 70)
print("✅ CUSTOM TRAINER DEFINED")
print("=" * 70)
print("Features:")
print("   ✓ Loss tracking (every 50 iterations)")
print("   ✓ Local checkpoint (every 400 iterations)")
print("   ✓ Checkpoint backup (every 400 iterations) ⚡")
print("   ✓ Validation (every 500 iterations)")
print("   ✓ Session crash protection 🛡️")
print("=" * 70)

---
## 🚀 STEP 8: Start Training

**IMPORTANT:** 
- Training runs for 10,000 iterations (~2-3 hours)
- Checkpoints saved every 400 iterations to `/kaggle/working/output/`
- Backups to `/kaggle/working/checkpoints/` every 400 iterations
- If session crashes, you can resume from last checkpoint!

In [ ]:
# ========================================
# STEP 8: START TRAINING
# ========================================

print("=" * 70)
print("🚀 STARTING TRAINING")
print("=" * 70)
print(f"📊 Training images: {len(train_dicts)}")
print(f"📊 Validation images: {len(val_dicts)}")
print(f"📊 Total iterations: {cfg.SOLVER.MAX_ITER:,}")
print(f"📊 Batch size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"📊 Approx epochs: ~{(cfg.SOLVER.MAX_ITER * cfg.SOLVER.IMS_PER_BATCH) / len(train_dicts):.1f}")
print("=" * 70)
print("\n💾 Auto-save schedule:")
print(f"   - Local checkpoint: every {cfg.SOLVER.CHECKPOINT_PERIOD} iterations")
print(f"   - Checkpoint backup: every 400 iterations")
print(f"   - Validation: every {cfg.TEST.EVAL_PERIOD} iterations")
print("\n⏱️  Training will take 2-3 hours on T4 GPU")
print("=" * 70)

# Create trainer
trainer = ImprovedTrainer(cfg)

# Check for existing checkpoints
existing_checkpoints = sorted(CHECKPOINT_DIR.glob("model_*.pth"))
if existing_checkpoints:
    print(f"\n📦 Found {len(existing_checkpoints)} checkpoints:")
    for ckpt in existing_checkpoints[-3:]:
        print(f"   - {ckpt.name}")
    
    # Ask if want to resume (you can manually set this)
    RESUME_FROM_CHECKPOINT = False  # Set to True to resume
    
    if RESUME_FROM_CHECKPOINT:
        latest_ckpt = existing_checkpoints[-1]
        print(f"\n🔄 Resuming from: {latest_ckpt.name}")
        shutil.copy(latest_ckpt, OUTPUT_DIR / latest_ckpt.name)
        trainer.resume_or_load(resume=True)
    else:
        print("\n🆕 Starting fresh training from COCO weights")
        trainer.resume_or_load(resume=False)
else:
    print("\n🆕 No checkpoints found. Starting fresh training.")
    trainer.resume_or_load(resume=False)

print("\n" + "=" * 70)
print("🏃 TRAINING IN PROGRESS...")
print("=" * 70)

# Start training
try:
    trainer.train()
    
    print("\n" + "=" * 70)
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 70)
    
    # Save final model to checkpoints
    final_model = OUTPUT_DIR / "model_final.pth"
    if final_model.exists():
        final_backup = CHECKPOINT_DIR / "model_final.pth"
        shutil.copy(final_model, final_backup)
        print(f"💾 Final model saved to: {final_backup}")
    
    # Save losses
    losses_history = trainer.loss_hook.losses
    print(f"📊 Completed {len(losses_history)} iterations")
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")
    print(f"💾 Latest checkpoint saved in: {OUTPUT_DIR}")
    print(f"💾 Checkpoint backups in: {CHECKPOINT_DIR}")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    print(f"💾 Check last checkpoint in: {OUTPUT_DIR}")
    raise

---
## 📊 STEP 9: View Training Metrics

Run this after training to see validation results.

In [ ]:
# ========================================
# STEP 9: VIEW VALIDATION METRICS
# ========================================

metrics_file = OUTPUT_DIR / "metrics.json"

if metrics_file.exists():
    print("=" * 70)
    print("📈 VALIDATION RESULTS")
    print("=" * 70)
    
    # Load metrics
    metrics_list = []
    with open(metrics_file, 'r') as f:
        for line in f:
            metrics_list.append(json.loads(line))
    
    # Filter validation metrics
    val_metrics = [m for m in metrics_list if 'bbox/AP' in m]
    
    if val_metrics:
        print(f"\n✅ Found {len(val_metrics)} validation checkpoints\n")
        
        # Show last 5 validations
        for m in val_metrics[-5:]:
            iteration = m.get('iteration', 'N/A')
            print(f"\n📊 Iteration {iteration}:")
            print(f"   bbox/AP     (IoU=0.50:0.95): {m.get('bbox/AP', 0):.2f}%")
            print(f"   bbox/AP50   (IoU=0.50):      {m.get('bbox/AP50', 0):.2f}%")
            print(f"   bbox/AP75   (IoU=0.75):      {m.get('bbox/AP75', 0):.2f}%")
            if 'segm/AP' in m:
                print(f"   segm/AP     (IoU=0.50:0.95): {m.get('segm/AP', 0):.2f}%")
                print(f"   segm/AP50   (IoU=0.50):      {m.get('segm/AP50', 0):.2f}%")
        
        # Best result
        best = max(val_metrics, key=lambda x: x.get('bbox/AP', 0))
        print("\n" + "=" * 70)
        print("🏆 BEST VALIDATION RESULT")
        print("=" * 70)
        print(f"   Iteration: {best.get('iteration', 'N/A')}")
        print(f"   bbox/AP:   {best.get('bbox/AP', 0):.2f}%")
        print(f"   segm/AP:   {best.get('segm/AP', 0):.2f}%")
        print("=" * 70)
        
        print("\n💡 Performance Guide:")
        print("   bbox/AP > 30%: Good")
        print("   bbox/AP > 35%: Very Good")
        print("   bbox/AP > 40%: Excellent")
        print("=" * 70)
    else:
        print("⚠️  No validation metrics found")
else:
    print("❌ Metrics file not found. Complete training first!")

---
## 📉 STEP 10: Visualize Training Losses

Plot loss curves to see training progress.

In [ ]:
# ========================================
# STEP 10: VISUALIZE LOSSES
# ========================================

if 'trainer' in locals() and hasattr(trainer, 'loss_hook'):
    losses_history = trainer.loss_hook.losses
    
    if len(losses_history) > 0:
        df_losses = pd.DataFrame(losses_history)
        
        # Add total loss
        loss_cols = [c for c in df_losses.columns if 'loss' in c]
        df_losses['total_loss'] = df_losses[loss_cols].sum(axis=1)
        
        # Plot
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle('Training Loss Evolution', fontsize=16, fontweight='bold')
        
        loss_names = [
            'loss_cls',
            'loss_box_reg',
            'loss_mask',
            'loss_rpn_cls',
            'loss_rpn_loc',
            'total_loss'
        ]
        
        for idx, loss_name in enumerate(loss_names):
            ax = axes[idx // 3, idx % 3]
            
            if loss_name in df_losses.columns:
                ax.plot(df_losses.index, df_losses[loss_name], linewidth=2, color='#2E86AB')
                ax.set_title(loss_name.replace('_', ' ').title(), fontsize=14, fontweight='bold')
                ax.set_xlabel('Iteration', fontsize=12)
                ax.set_ylabel('Loss', fontsize=12)
                ax.grid(True, alpha=0.3)
                ax.set_facecolor('#F8F9FA')
            else:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=12)
                ax.set_title(loss_name.replace('_', ' ').title(), fontsize=14)
        
        plt.tight_layout()
        
        # Save
        plot_path = OUTPUT_DIR / 'training_losses.png'
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f"✅ Loss plot saved to: {plot_path}")
        
        # Copy to checkpoints
        checkpoint_plot = CHECKPOINT_DIR / 'training_losses.png'
        shutil.copy(plot_path, checkpoint_plot)
        print(f"💾 Backup saved to: {checkpoint_plot}")
        
        plt.show()
        
        # Statistics
        print("\n" + "=" * 70)
        print("📊 LOSS STATISTICS")
        print("=" * 70)
        
        first_10 = df_losses.head(10).mean()
        last_10 = df_losses.tail(10).mean()
        
        print("\n🔴 First 10 iterations (average):")
        for loss_name in loss_names:
            if loss_name in first_10:
                print(f"   {loss_name:20s}: {first_10[loss_name]:.4f}")
        
        print("\n🟢 Last 10 iterations (average):")
        for loss_name in loss_names:
            if loss_name in last_10:
                print(f"   {loss_name:20s}: {last_10[loss_name]:.4f}")
        
        print("\n📉 Improvement:")
        for loss_name in loss_names:
            if loss_name in first_10 and loss_name in last_10:
                improvement = ((first_10[loss_name] - last_10[loss_name]) / first_10[loss_name]) * 100
                print(f"   {loss_name:20s}: {improvement:+.1f}%")
        
        print("=" * 70)
    else:
        print("❌ No loss history found")
else:
    print("❌ Trainer not found. Run training first!")

---
## 🔄 HOW TO RESUME TRAINING (If Session Crashes)

If your Kaggle session crashes or times out, **you haven't lost your work!**

### Steps to Resume:

1. **Re-run all setup cells** (Steps 1-7)

2. **In Step 8**, set `RESUME_FROM_CHECKPOINT = True`:
   ```python
   RESUME_FROM_CHECKPOINT = True  # Change this to True
   ```

3. **The trainer will automatically:**
   - Find your latest checkpoint in `/kaggle/working/checkpoints/`
   - Copy it to output directory
   - Resume training from that iteration
   - Continue to iteration 10,000

### Your Checkpoints:

Check your checkpoints folder:
```
/kaggle/working/checkpoints/
├── model_0000400.pth  (iteration 400)
├── model_0000800.pth  (iteration 800)
├── model_0001200.pth  (iteration 1200)
└── ... (every 400 iterations)
```

**Even if you reached 9599/10000 and crashed, you have a checkpoint at 9600!** 🎉

**Important:** Download your checkpoints regularly from Kaggle output folder to avoid losing them!

---
## 📝 SUMMARY

### What This Notebook Does:

1. ✅ Sets up GPU environment (Kaggle)
2. ✅ Configures Kaggle dataset paths
3. ✅ Extracts Cityscapes dataset from ZIPs
4. ✅ Loads custom dataset with 8 classes
5. ✅ Configures Mask R-CNN with optimal hyperparameters
6. ✅ **Saves checkpoint every 400 iterations**
7. ✅ **Backs up to checkpoints folder every 400 iterations**
8. ✅ Validates on val set every 500 iterations
9. ✅ Trains for 10,000 iterations (~2-3 hours)
10. ✅ Visualizes training progress

### Key Protection Features:

- 💾 **Local checkpoints**: Every 400 iterations
- 💾 **Checkpoint backups**: Every 400 iterations  
- 🛡️ **Crash protection**: Never lose more than 400 iterations of work
- 🔄 **Easy resume**: One-click resume from any checkpoint
- ⬇️ **Download**: Save checkpoints from Kaggle output regularly

### Expected Results:

After 10,000 iterations, you should achieve:
- **bbox/AP**: 35-40% (object detection)
- **segm/AP**: 30-35% (instance segmentation)

Much better than the 3,000-iteration training! 🚀

### Kaggle Tips:

- Monitor GPU usage in right sidebar
- Download checkpoints regularly (every 2000-3000 iterations)
- Enable "Always save output" in Kaggle settings
- GPU time limit: ~30 hours/week (plenty for this training!)